## Init

In [0]:
import requests
import json
from datetime import datetime
from pyspark.sql.functions import current_timestamp, lit

from bronze_config import cities_config

## Read from API 

In [0]:
raw_records = []

# Iterate over the cities and query Open-Meteo
for c in cities_config:
    url = (
        f"https://api.open-meteo.com/v1/forecast?"
        f"latitude={c['lat']}&longitude={c['lon']}"
        f"&hourly=temperature_2m,relative_humidity_2m,precipitation"
        f"&past_days=1&forecast_days=1&forecast_hours=1"
        f"&timezone=America%2FMexico_City"
    )
    response = requests.get(url)
    
    if response.status_code == 200:
        data = response.json()
        raw_records.append({
            "city_name": c["city"],
            "state_name": c["state"],
            "raw_payload": json.dumps(data),
            "ingestion_timestamp": datetime.now().isoformat()
        })

# Convert to DataFrame from PySpark
df = spark.createDataFrame(raw_records)

## Write in bronze table

In [0]:
(
    df.write
        .mode("append")
        .format("delta")
        .saveAsTable("weather.bronze.data_raw")
)